In [1]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## Summarize messages

In [2]:
summarization_model = create_noreason_llm(model="databricks-gemma-3-12b")
pg_checkpointer = create_pg_checkpointer()


agent = create_agent(
    model=llm_noreason, 
    checkpointer=pg_checkpointer,
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": f"{new_conversation_id()}"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nTo roleplay a scenario involving a lunar colony called Lunapolis, including details about its capital, weather, population of cheese miners, and potential labor disputes.\n\n## SUMMARY\n\nThe scenario is set on the moon, where the capital city is Lunapolis. Lunapolis experiences extreme temperatures (high of 120C, low of -100C) and is populated by 100,000 cheese miners. These miners are considering a strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nContinue roleplaying the scenario, potentially exploring the reasons for the miners' dissatisfaction, the president's response, or the impact of a potential strike on Lunapolis.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='57440160-dc44-4802-be09-dd1028472eea'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond 

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

To roleplay a scenario involving a lunar colony called Lunapolis, including details about its capital, weather, population of cheese miners, and potential labor disputes.

## SUMMARY

The scenario is set on the moon, where the capital city is Lunapolis. Lunapolis experiences extreme temperatures (high of 120C, low of -100C) and is populated by 100,000 cheese miners. These miners are considering a strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

Continue roleplaying the scenario, potentially exploring the reasons for the miners' dissatisfaction, the president's response, or the impact of a potential strike on Lunapolis.


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    filtered_messages = [RemoveMessage(id=m.id) for m in tool_messages]
    return {"messages": filtered_messages}

In [6]:
agent = create_agent(
    model=llm_noreason,
    checkpointer=pg_checkpointer,
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": f"{new_conversation_id()}"}}
)

for msg in response["messages"]:
    print(f"{msg.type}: {msg.content}")

human: My device won't turn on. What should I do?
ai: Is the device plugged in and turned on?
human: Yes, it's plugged in and turned on.
ai: Is the device showing any lights or indicators?
human: What's the temperature of the device?
ai: I cannot physically feel or measure the temperature of your device. You would need to touch it (carefully) or check if it has a built-in temperature sensor display.

However, knowing the temperature is actually a very good diagnostic step:

*   **If it feels extremely hot:** It might be overheating, which can cause the device to shut down automatically to protect itself. In this case, unplug it, let it cool down completely in a well-ventilated area, and try turning it on again.
*   **If it feels ice cold:** This is less common but could indicate a battery issue or that the device has been in a freezing environment, which can temporarily prevent it from powering on.
*   **If it feels normal:** The issue is likely related to the battery, the power button

`trim_messages` is a LangChain utility that shortens a chat history so it fits a token budget before sending it to a model.

In this cell, it is configured as:

- `max_tokens=65`: target limit for total message tokens.
- `strategy="last"`: keep the **most recent** messages and drop older ones first.
- `token_counter="approximate"`: use a fast approximate token count (not exact model tokenization).
- `include_system=True`: preserve `SystemMessage` content when possible.
- `allow_partial=False`: do **not** keep partial/truncated messages; each kept message must be whole.
- `start_on="human"`: trimming boundary starts from a human turn, helping keep conversation alignment by beginning with user context.

### What the `"last"` strategy means
With `"last"`, the trimmer walks backward from the end of the conversation and keeps the newest exchanges until adding another message would exceed `max_tokens`. This is useful when recent context is more important than older history.

In [ ]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=65,
    strategy="last", 
    token_counter="approximate",
    include_system=True,
    allow_partial=False, 
    start_on="human", 
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="what's 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="what's 2 + 2", additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## `filter_messages` in LangChain

`filter_messages` selects only the messages you want from a chat history and returns a new list.

- Useful for cleaning context before sending messages to a model.
- Can filter by **message type** (e.g., `"human"`, `"ai"`, `"system"`).
- Can also filter using metadata like names/IDs (when provided).

### Quick example
```python
filter_messages(messages, include_types="human")
```

This keeps only `HumanMessage` entries and removes system/AI messages.


In [9]:
from langchain_core.messages import SystemMessage, filter_messages, HumanMessage, AIMessage

messages = [
    SystemMessage("you are a good assistant", id="1"),
    HumanMessage("example input", id="2", name="example_user"),
    AIMessage("example output", id="3", name="example_assistant"),
    ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
    HumanMessage("real input", id="4", name="bob"),
    AIMessage("real output", id="5", name="alice"),
]


In [10]:

filter_messages(messages, include_types="human")

[HumanMessage(content='example input', additional_kwargs={}, response_metadata={}, name='example_user', id='2'),
 HumanMessage(content='real input', additional_kwargs={}, response_metadata={}, name='bob', id='4')]

In [11]:
filter_messages(messages, include_types={"human", "tool"})

[HumanMessage(content='example input', additional_kwargs={}, response_metadata={}, name='example_user', id='2'),
 ToolMessage(content='temp=42C voltage=2.9v … greeble complete.', tool_call_id='2'),
 HumanMessage(content='real input', additional_kwargs={}, response_metadata={}, name='bob', id='4')]

### Rebuilding our custom middleware using langchain methods ...

In [12]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]
    filtered_messages = filter_messages(messages, include_types={"human", "tool"})

    return {"messages": filtered_messages}